# 00 — Data Sanity Check

**Purpose:** Verify that the raw Credit Card Fraud dataset loads correctly, matches its expected schema, and that the reusable `load_raw()` function works end-to-end.

**Pass criteria:** All assertions below pass without error. If any fail, stop and fix before running `01_eda.ipynb`.

**Dataset:** [Credit Card Fraud Detection — Kaggle (ULB)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

**Expected shape:** `(284807, 31)`

**Expected fraud count:** `492` (≈0.173%)

In [1]:
import sys
from pathlib import Path

# Project root = parent of the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent

# Make project root importable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Standard imports
import pandas as pd
import numpy as np

# Path to the raw dataset
RAW_CSV_PATH = PROJECT_ROOT / "data" / "raw" / "creditcard.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw CSV path: {RAW_CSV_PATH}")
print(f"CSV exists: {RAW_CSV_PATH.exists()}")

Project root: c:\Users\ADULT\Downloads\fraud-detector
Raw CSV path: c:\Users\ADULT\Downloads\fraud-detector\data\raw\creditcard.csv
CSV exists: True


In [2]:
df = pd.read_csv(RAW_CSV_PATH)
df.head()



,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [3]:
# Shape
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print()

# Dtypes — critical: confirms our data contract
print("Dtypes:")
print(df.dtypes)
print()

# Null check — headline first, details only if needed
total_nulls = df.isnull().sum().sum()
print(f"Total null values: {total_nulls}")


Shape: (284807, 31)
Rows: 284,807
Columns: 31

Dtypes:
Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object

Total null values: 0


In [4]:
# Class distribution
counts = df['Class'].value_counts()
n_legit = counts[0]
n_fraud = counts[1]
total = n_legit + n_fraud

# Percentages (divide by total, not by each other)
legit_pct = n_legit / total * 100
fraud_pct = n_fraud / total * 100

# Imbalance ratio — "1 fraud per X legit"
imbalance_ratio = n_legit / n_fraud

print(f"Legit transactions: {n_legit:,} ({legit_pct:.4f}%)")
print(f"Fraud transactions: {n_fraud:,} ({fraud_pct:.4f}%)")
print(f"Imbalance ratio:    1 fraud per {imbalance_ratio:.0f} legitimate transactions")



Legit transactions: 284,315 (99.8273%)
Fraud transactions: 492 (0.1727%)
Imbalance ratio:    1 fraud per 578 legitimate transactions


In [5]:
# Data contract enforcement — if any of these fail, stop everything
assert df.shape == (284807, 31), f"Wrong shape: expected (284807, 31), got {df.shape}"

assert df.isnull().sum().sum() == 0, "Unexpected nulls in dataset"

assert df['Class'].sum() == 492, f"Wrong fraud count: expected 492, got {df['Class'].sum()}"

assert set(df['Class'].unique()) == {0, 1}, f"Unexpected class values: {df['Class'].unique()}"

assert df['Amount'].min() >= 0, f"Negative transaction amount found: {df['Amount'].min()}"

print("All data contract assertions passed.")

All data contract assertions passed.


In [6]:
from fraud.data_loader import load_raw

df_via_func = load_raw()

assert df.equals(df_via_func), "Data mismatch between direct load and function load"
print(f"load_raw() returned DataFrame with shape {df_via_func.shape}")
print("Function and inline load produce identical data.")

load_raw() returned DataFrame with shape (284807, 31)
Function and inline load produce identical data.
